# Find the cheapest model that still does the job

In this guide, you'll run the same evaluator under a few model choices and keep the cheapest model
that still catches the bad output. The sample task is a visual QA check that flags broken
infographic tiles. The same pattern works for classifiers, extractors, routers, and other repeated
steps where cost matters.

## Setup

Load the launch helpers.

In [ ]:
import pathlib
import sys


def _find_visual_artifact_example_root():
    cwd = pathlib.Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.append(base)
        candidates.append(base / "examples" / "notebooks" / "visual_artifact")
    for candidate in candidates:
        if (candidate / "shepherd_usecases" / "visual_artifact" / "launch.py").exists():
            return candidate
    raise RuntimeError(
        "Cannot find examples/notebooks/visual_artifact. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    )


example_root = _find_visual_artifact_example_root()
if str(example_root) not in sys.path:
    sys.path.insert(0, str(example_root))
try:
    from shepherd_usecases.visual_artifact import launch, viz
except Exception as exc:
    raise RuntimeError(
        "Could not import the visual-artifact notebook helpers. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    ) from exc

launch.bootstrap(example_root=example_root)

## What this run does

Right-sizing runs the same evaluator under each model choice, grades every run against a
deterministic oracle, then keeps the cheapest model that still caught the bad tile.

## The input

The run starts with one prompt and a few model choices. `model_choices` maps each tier name to a
model identifier; only the model changes between evaluator runs.

In [ ]:
prompt = launch.default_prompt()
model_choices = launch.model_choices()
model_choices

## Run Shepherd

Open a workspace for this comparison. Every evaluator run and the selector attach to it.

In [ ]:
workspace = launch.open_workspace("model-right-sizing-lab", prompt=prompt, metadata={"usecase": "model-right-sizing-lab"})

Run the evaluator once per model choice and grade the results. Each run goes through
`launch.run_static` with a different model tier; `grade_runs` checks each verdict against a
deterministic oracle — a check that already knows which tile is broken — and marks each run as
passed or missed.

In [ ]:
runs = {}
for config_name, model in model_choices.items():
    runs[config_name] = launch.run_static(
        workspace,
        name=f"rightsize-{config_name}",
        output_path=launch.VERDICT_PATH,
        output_content=launch.evaluator_content(config_name, model),
        model=model,
        metadata={"model_tier": config_name, "model": model},
    )

graded = launch.grade_runs(runs)
viz.show(viz.table([
    {
        "config": name,
        "model": item.model,
        "cost": item.cost,
        "catches hard fail": item.catches_hard_fail,
        "state": "passed" if item.passed else "missed",
    }
    for name, item in graded.items()
]))

With every run graded, run the selector. It receives each run's verdict as an artifact ref in
`verdict_refs` and reads them to pick the cheapest model that passed. `read_json` retrieves the
selector's decision from `DECISION_PATH`.

In [ ]:
verdict_refs = {
    f"verdict_{name}": launch.artifact_ref(item.run, launch.VERDICT_PATH, label=name)
    for name, item in graded.items()
}
selector = launch.run_with_artifact_refs(
    workspace,
    name="selector",
    refs=verdict_refs,
    output_path=launch.DECISION_PATH,
    output_content=launch.selector_content(graded),
    after=[item.run for item in graded.values()],
)
decision = launch.read_json(selector, launch.DECISION_PATH)
decision

Mark the winning run as selected, discard the rest, and release the selector's output.

In [ ]:
for name, item in graded.items():
    if name == decision["kept"]:
        item.run.output().select()
    else:
        item.run.output().discard()
selector.output().release()

## Trace

How does Shepherd run one task under many models and let a selector read them all? The trace below
captures every action an agent took while executing the tasks above, plus the relationships among
those actions. Click the nodes to learn more about the events.

In [ ]:
viz.show(viz.trace(workspace.flow, runs | {"selector": selector}, height="620px", detail="events"))

In [ ]:
workspace.close()

## Want to understand how it works?

If you want to understand how this works in greater detail, open the
[Model Right-Sizing internals notebook](model_right_sizing_internals.ipynb).

## Next steps

- [Find the best approach by trying several alternatives](visual_variant_studio.ipynb)
- [Recover a pipeline when one step drifts](visual_pipeline_recovery.ipynb)